# Séance 7 : Bases vectorielles (Embeddings, similarité cosinus, ANN, HNSW)

**Enseignant :** Jean Delpech  
**Cours :** Algorithmie et développement dans l'ingénierie des données  
**Classe :** M1 Data  
**Année scolaire :** 2025/2026  
**Dernière mise à jour :** juin 2026

## Objectifs

- Comprendre ce qu'est un embedding et pourquoi la proximité géométrique reflète la similarité sémantique
- Calculer et interpréter la similarité cosinus
- Comprendre la malédiction de la dimensionnalité et pourquoi les index exacts échouent en haute dimension
- Saisir le principe de HNSW et son lien avec le BFS (séance 4) et les heaps (séance 3)
- Connaître les outils de référence : Faiss et pgvector

## Plan

| # | Partie |
|---|---|
| 1 | De la représentation symbolique à la représentation vectorielle |
| 2 | Similarité cosinus |
| 3 | La malédiction de la dimensionnalité |
| 4 | Approximate Nearest Neighbor (ANN) |
| 5 | HNSW : Hierarchical Navigable Small World |
| 6 | Faiss et pgvector : panorama des outils |
| 7 | Pour aller plus loin : lien avec les mini-mémoires |

> **Note pédagogique**  
> Ce notebook est volontairement théorique. Les algorithmes sont présentés sous forme de pseudo-code commenté et de descriptions d'architecture. L'implémentation *from scratch* constitue le travail attendu dans le cadre des mini-mémoires.

> **Contexte dans le module**  
> Cette séance s'appuie directement sur la séance 6 (index textuels, TF-IDF, BM25) dont elle prolonge la représentation vectorielle, et sur la séance 4 (graphes, BFS) dont HNSW est une application directe. La similarité cosinus est déjà apparue dans la séance 6 comme mesure de pertinence dans le modèle vectoriel TF-IDF.

# PARTIE 1 : De la représentation symbolique à la représentation vectorielle

## 1.1 La limite fondamentale de l'approche lexicale

En séance 6, nous avons vu que TF-IDF représente chaque document comme un **vecteur creux** (*sparse*) dans un espace de dimension égale à la taille du vocabulaire : chaque dimension correspond à un terme, la valeur exprime son poids dans le document.

Cette représentation a une limite fondamentale : elle est **purement lexicale**. Deux documents parlant exactement du même sujet mais avec des mots différents sont perçus comme totalement distincts par TF-IDF. Les exemples sont nombreux :

- `"voiture"` et `"automobile"` : quasi-synonymes, aucun lien dans l'espace TF-IDF
- `"chien"` et `"labrador"` : relation générique/spécifique, invisible pour TF-IDF
- `"ce produit est formidable"` et `"j'adore ce produit"` : même sens, aucun mot en commun

La recherche booléenne et BM25 ne peuvent pas retrouver un document qui décrit exactement ce qu'on cherche si les mots utilisés sont différents. Ce n'est pas un défaut d'implémentation : c'est une **limite structurelle** du modèle *bag-of-words*.

## 1.2 L'idée des embeddings : encoder la sémantique dans la géométrie

L'idée des **embeddings** (représentations vectorielles denses) est radicalement différente : plutôt que de représenter un objet par les mots qui le décrivent, on l'encode comme un **point dans un espace géométrique** où les objets sémantiquement similaires sont physiquement proches.

Un **embedding** est une représentation dense d'un objet (mot, phrase, document, image, entité…) sous la forme d'un vecteur de réels de dimension fixe $d$ :

$$\text{objet} \rightarrow \mathbf{v} \in \mathbb{R}^d$$

La propriété fondamentale, apprise lors de l'entraînement du modèle :

> **Deux objets sémantiquement similaires ont des vecteurs géométriquement proches.**

C'est une propriété **apprise**, pas définie à la main. Le modèle d'embedding est entraîné sur de grandes quantités de texte (ou d'images, ou de graphes…) de telle sorte que deux objets qui apparaissent souvent dans des contextes similaires se retrouvent proches dans l'espace vectoriel.

## 1.3 Comment obtient-on ces vecteurs ?

Ok, on veut représenter les vecteurs dans un espace avec un certain nombre de dimensions, où les vecteurs seraient plutôt denses, à l’inverse de TF-IDF. Mais si les dimensions n'ont pas de signification individuelle lisible, comment le modèle les construit-il ?

Le principe commun à tous les modèles d'embedding est une idée formulée par le linguiste [John Firth](https://en.wikipedia.org/wiki/John_Rupert_Firth) en 1957, souvent citée sous la forme :
> « You shall know a word by the company it keeps. » et «  "a word is characterized by the company it keeps"

Autrement dit : le sens d'un mot se déduit des contextes dans lesquels il apparaît. « Chien » et « labrador » apparaissent souvent dans les mêmes contextes (on les promène, ils aboient, ils ont des maîtres…), donc ils doivent avoir des représentations proches. « Chien » et « budget » n'apparaissent presque jamais ensemble, leurs représentations doivent être éloignées. C'est la structure statistique du corpus qui définit la géométrie de l'espace.

### Exemple : Le mécanisme de `Word2Vec`
`Word2Vec` ([Mikolov et al., 2013](https://arxiv.org/pdf/1301.3781)) est l'exemple le plus simple pour comprendre le principe général. Le modèle entraîne un réseau de neurones à prédire le contexte d'un mot (ou inversement, à prédire un mot depuis son contexte). Pour le mot "chien" dans la phrase "le chien court dans le jardin", le modèle doit apprendre à prédire que ses voisins sont "court", "jardin", "le".

Voilà un schéma qui illustre ce principe mis en œuvre par `Word2Vec` (variante Skip-gram) :

![Schéma : la matrice poids Word2Vec encode le contexte d’un mot](./Images/Embedding-Vector.png)

Les vecteurs d'embedding ne sont pas l'objectif du modèle : ils sont les poids de la couche cachée du réseau, appris comme sous-produit de la tâche de prédiction. À la fin de l'entraînement, deux mots qui apparaissent dans des contextes similaires ont des poids similaires, c'est-à-dire que ce sont des vecteurs proches dans l’espace obtenu.

### La dimension dd
`d`, la dimension du vecteur, est un hyperparamètre choisi avant l'entraînement. Elle n'est pas liée au vocabulaire. C'est précisément pourquoi les dimensions n'ont pas de signification individuelle : elles ne correspondent pas à des concepts prédéfinis, elles sont le résultat d'une compression statistique.

### Généralisation aux modèles contextuels (BERT et suivants)
`Word2Vec` produit un vecteur fixe par mot, indépendant du contexte : le "chien" meilleur ami de l’homme et le "chien" d’une arme à feu ont le même vecteur. BERT et ses successeurs corrigent cette limite en produisant un vecteur différent selon le contexte de la phrase entière, le même mécanisme de fond (apprendre depuis les co-occurrences), mais avec une architecture transformer qui tient compte de toute la phrase simultanément grâce à un mécanisme attentionnel.
Pour les embeddings de phrase (*sentence-transformers*), le principe est similaire mais la tâche d'entraînement change : le modèle apprend à rapprocher des paires de phrases sémantiquement équivalentes et à éloigner des paires non liées (entraînement par *[contrastive loss](https://fr.wikipedia.org/wiki/Fonction_de_co%C3%BBt_par_triplet)*). Le résultat est un unique vecteur représentant une phrase entière plutôt qu'un mot.

On rentre dans des considérations très abstraites et très concrètes, pour votre culture, retenez simplement que les vecteurs d'embedding ne sont pas construits à la main ni déduits de règles linguistiques. Ils émergent de l'optimisation d'une tâche de prédiction sur un très grand corpus. La géométrie de l'espace (quels vecteurs sont proches, quels angles existent entre eux) est entièrement déterminée par la structure statistique des données d'entraînement. Cet espace encode des relations sémantiques réelles, mais pour la même raison il encode aussi les biais présents dans les données.

## 1.4 Exemples de modèles d'embedding

| Modèle | Dimension $d$ | Domaine | Remarque |
|---|---|---|---|
| Word2Vec | 100 – 300 | Mots | Premier modèle populaire (Google, 2013) |
| GloVe | 100 – 300 | Mots | Approche par co-occurrence globale |
| BERT (base) | 768 | Phrases / documents | Contextuel : un même mot a des embeddings différents selon le contexte |
| RoBERTa | 768 | Phrases / documents | Variante de BERT, entraînement amélioré |
| `all-MiniLM-L6-v2` | 384 | Phrases | Modèle léger, usage courant (sentence-transformers) |
| OpenAI `ada-002` | 1536 | Texte général | Embedding commercial via API |
| CLIP | 512 | Images et texte | Espace commun image/texte (OpenAI, 2021) |

La dimension $d$ est un paramètre de conception du modèle : une dimension plus élevée permet en théorie de capturer des nuances plus fines, au prix d'une consommation mémoire et d'un coût de calcul plus importants.

## 1.5 TF-IDF vs embedding

Ce tableau récapitule la différence structurelle qui existe entre une approche TF-IDF et l’embedding en matière de « densité » des vecteurs obtenus :

| Critère | TF-IDF | Embedding |
|---|---|---|
| Nature | Vecteur **creux** (*sparse*) | Vecteur **dense** |
| Dimension | = taille du vocabulaire (50 000+) | Fixe (300 – 1536) |
| Valeurs non nulles | Quelques dizaines | Toutes les dimensions |
| Espace de représentation | Espace des **termes** | Espace **sémantique latent** |
| Synonymes | Aucun lien | Vecteurs proches |
| Construction | Formule statistique | Entraînement d'un réseau de neurones |
| Interprétabilité | Chaque dimension a un sens (= un terme) | Dimensions sans sens individuel |

> Retenez que ce que l’on appelle **l'espace sémantique latent** est le concept central ici. Contrairement à TF-IDF où chaque dimension représente un terme précis du vocabulaire, les dimensions d'un embedding n'ont pas de signification individuelle lisible. C'est l'arrangement collectif de toutes les dimensions qui encode le sens. On parle de représentation *latente* car la structure sémantique est implicite, encodée dans les relations entre vecteurs plutôt que dans les composantes elles-mêmes.

## 1.6 Une arithmétique avec des mots

Les premiers modèles Word2Vec ont mis en évidence une propriété frappante des embeddings : l'arithmétique vectorielle dans l'espace sémantique correspond à des relations sémantiques.

L'exemple classique (Mikolov et al., 2013) :

$$\text{vec}(\text{"roi"}) - \text{vec}(\text{"homme"}) + \text{vec}(\text{"femme"}) \approx \text{vec}(\text{"reine"})$$

Ce n'est pas une coïncidence : c'est une conséquence directe du fait que l'entraînement a appris à encoder les *relations sémantiques* comme des *directions* dans l'espace vectoriel. La direction `"roi" → "reine"` est la même que la direction `"homme" → "femme"`.

> Cet exemple, qui frappe si clairement les esprits qu’il est présent dans tous les cours sur le sujet, [est à relativiser selon ce court article de Mc Cheng](https://medium.com/@mc_cheng/king-man-woman-is-not-queen-9cbf0afe6c72). Néanmoins, il ne faut pas tomber dans le travers inverse, comme le fait l’article de Mc Cheng avec un titre qui laisse entendre que l’idée derrière l’équation est totalement fausse.  En pratique, le vecteur résultant est souvent plus proche de "roi" que de "reine" si l'on n'exclut pas le mot source du calcul de similarité. Cette exclusion est une convention documentée dès le papier original (Mikolov et al., 2013). C’est un peu fort de suggérer que ce serait une manipulation « pour que ça marche ». Il faut juste garder à l’esprit que l'analogie n'est pas aussi « automatique » qu'on le présente souvent. Cette propriété est en fait très dépendante du modèle et du corpus d'entraînement. Il n’en est pas moins vrai que les modèles d'embedding structurent l'espace de façon à encoder des relations sémantiques sous forme de directions vectorielles. Que "reine" soit 1re ou 2e ne remet pas en cause ce principe.
>
> Le fait que le résultat soit plus proche de `king` que de `queen` s'explique géométriquement : dans un espace vectoriel, `king − man + woman` reste dans un voisinage de `king` car on n'a soustrait qu'une partie de ce qui distingue `king` de ses voisins. C'est une propriété mathématique normale, et l'exclusion du terme source est une convention de calcul d'analogie standard développée depuis l’article princeps de Mikolov et al..

D'autres analogies que vous pouvez essayer de vérifier empiriquement :
- `vec("Paris") - vec("France") + vec("Allemagne") ≈ vec("Berlin")`
- `vec("marcher") - vec("marché") ≈ vec("courir") - vec("couru")`

On peut déduire de ces exemples que les embeddings ne se contentent pas d'associer des vecteurs à des mots : ils **structurent l'espace** de façon à refléter la structure sémantique du langage.

# PARTIE 2 : Similarité cosinus

## 2.1 Distance euclidienne : ça ne marche pas !

Pour mesurer la proximité entre deux embeddings, on pourrait utiliser la **distance euclidienne** classique :

$$d_E(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\| = \sqrt{\sum_{i=1}^d (u_i - v_i)^2}$$

Cette mesure est sensible à la **norme** (la « longueur ») des vecteurs. Deux vecteurs qui pointent dans la même direction mais avec des normes très différentes auront une grande distance euclidienne, alors qu'ils encodent la même orientation sémantique.

![Illustration norme euclidienne](./Images/NormeEuclidienne.png)

En pratique, la norme d'un embedding dépend de facteurs non sémantiques : longueur du texte, fréquence d'apparition des tokens dans le corpus d'entraînement, etc. Elle ne reflète pas la sémantique. On veut une mesure qui capture uniquement l'**orientation** des vecteurs dans l'espace, pas leur magnitude.

## 2.2 Définition de la similarité cosinus

La **similarité cosinus** mesure le cosinus de l'angle $\theta$ entre deux vecteurs, indépendamment de leur norme :

$$\cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \cdot \|\mathbf{v}\|}$$

où $\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^d u_i v_i$ est le produit scalaire et $\|\mathbf{u}\| = \sqrt{\sum_{i=1}^d u_i^2}$ est la norme euclidienne. 

![Illustration Cosinus / produit scalaire](./Images/Cos-ProduitScalaire.png)

> En réalité on va utiliser le cosinus car c’est par la relation entre le cosinus et le produit scalaire qu’on définit un angle θ entre deux vecteurs sans faire appel à aucune autre construction géométrique propre à la dimension 2 ou 3. N’oublions pas qu’en matière *d’embeddings* on va se placer dans un espace avec beaucoup de dimensions et on a besoin des définitions rigoureuses de l’algébre, nos intuitions géométriques atteignants leurs limites.
>
> Ainsi la formule $\cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|\|\mathbf{v}\|}$ est valable dans $\mathbb{R}^d$
pour tout $d \geq 1$.

Propriétés :
- $\cos(\theta) = 1$ : vecteurs colinéaires de même sens → **« totalement » similaires**
- $\cos(\theta) = 0$ : vecteurs orthogonaux → **aucune relation**
- $\cos(\theta) = -1$ : vecteurs colinéaires de sens opposé → **« totalment » dissimilaires**

La **distance cosinus** est définie comme $1 - \cos(\theta) \in [0, 2]$. Elle vaut 0 pour des vecteurs identiques et constitue une métrique (à quelques nuances près) adaptée à la recherche de voisins.

## 2.3 Exemple de prise en compte de la norme

Considérons deux documents qui parlent du même sujet : un résumé de 100 mots et un article de 2000 mots sur le même thème. Leurs *embeddings* pointent dans la même direction sémantique, mais l'article long peut produire un vecteur de norme plus grande.

- Distance euclidienne : grande (les vecteurs sont loin l'un de l'autre)
- Similarité cosinus : proche de 1 (les vecteurs pointent dans la même direction)

La similarité cosinus capture correctement que les deux documents parlent du même sujet.

```
Exemple numérique en dimension 2 (illustration du principe):

doc_court = [1.0, 2.0]     (même direction sémantique)
doc_long  = [3.0, 6.0]     (même direction, norme 3× plus grande)
doc_autre = [2.0, 0.5]     (direction différente)
```
![Illustration exemple document court vs long vs autre](./Images/ExempleEmbeddingNorme.png)

```
cos(doc_court, doc_long)  = (1×3 + 2×6) / (√5 × √45) = 15 / 15 = 1.000
cos(doc_court, doc_autre) = (1×2 + 2×0.5) / (√5 × √4.25) ≈ 0.651

→ doc_court et doc_long sont perçus comme identiques : correct
→ doc_autre est perçu comme différent : correct
```

## 2.4 Produit scalaire et normalisation

En pratique, la plupart des modèles d'embedding produisent des vecteurs **normalisés** (norme 1). Dans ce cas, le produit scalaire $\mathbf{u} \cdot \mathbf{v}$ est directement égal à la similarité cosinus puisque $\|\mathbf{u}\| = \|\mathbf{v}\| = 1$ :

$$\cos(\theta) = \mathbf{u} \cdot \mathbf{v} \quad \text{si } \|\mathbf{u}\| = \|\mathbf{v}\| = 1$$

`Faiss` et `pgvector`,  les deux bibliothèques de référence pour l'indexation vectorielle que nous présenterons en partie 6, exploitent souvent cette normalisation pour remplacer les calculs de distance cosinus par de simples produits scalaires, ce qui est plus rapide à exécuter (une soustraction de moins par dimension).

## 2.5 Architecture d'un moteur de similarité par force brute

Avant d'aborder les structures d'index optimisées, posons le problème dans sa version naïve. Étant donné un corpus de $n$ embeddings et une requête $\mathbf{q}$, trouver les $k$ vecteurs les plus proches de $\mathbf{q}$.

### Quel est le problème à résoudre ? (=la stratégie, l’algorithme)

On a un corpus de $n$ embeddings et une requête $\mathbf{q}$. On veut trouver les $k$ vecteurs du corpus les plus similaires à $\mathbf{q}$ selon la similarité cosinus. 

C'est la définition du problème kNN (k-nearest neighbors) dans un espace vectoriel.
La stratégie naïve est immédiate : calculer la similarité cosinus entre $\mathbf{q}$ et chaque vecteur du corpus, puis retourner les $k$ meilleurs scores. C'est là que réside la force brute.

### Qu'est-ce qu'on veut pouvoir faire ? (=les méthodes)

Trois opérations suffisent à couvrir le cycle de vie de l'objet :

#### Alimenter le corpus. 

Il faut bien pouvoir ajouter des vecteurs. On va donc créer une méthode `add(vectors)`. On choisit une méthode séparée plutôt que de tout passer au constructeur pour pouvoir ajouter des vecteurs par lots successifs, ce qui est plus réaliste.

#### Calculer la similarité entre deux vecteurs.

C'est le cœur de notre problème. On en fait une méthode `cosine_similarity(u, v)` distincte de la méhtode qui va implémenter la recherche, pour deux raisons : 
1. d'abord pour respecter le principe de séparation des responsabilités. On aura une fonction `search` qui retournera les k-voisins. Pour ce faire elle bouclera sur le corpus et sélectionnera les meilleurs résultats, en déléguant le calcul de similarité à `cosine_similarity()`, ce qui sera plus facile à tester et à modifier. 
2. ensuite parce que cette `cosine_similarity()` sera utile indépendamment de `search` :on peut vouloir comparer deux embeddings ad hoc (ponctuellement), sans passer par la recherche dans un corpus, par exemple si on veut vérifier rapidement que deux phrases sont sémantiquement proches.

#### Chercher les k voisins
C'est l'opération principale, `search(query, k)`. Elle retourne pour chaque résultat l'identifiant du document dans le corpus (son numéro de position dans la liste) et la valeur de similarité cosinus calculée avec la requête, pas seulement l'identifiant seul. Si on ne retournait que les positions, on saurait quels documents sont les plus proches mais pas à quel point. Il faut qu’on soit capable de distinguer un résultat excellent (p. ex. similarité 0.97) d'un résultat médiocre (p. ex. similarité 0.51). La valeur de similarité est nécessaire pour toute décision en aval : seuiller les résultats sous un certain score, afficher un indicateur de confiance, ou re-classer les candidats, etc.

### De quoi a-t-on besoin pour faire ça ? (=les attributs)

En partant des méthodes :
* `search()` a besoin de parcourir tous les vecteurs du corpus. Il faut donc stocker le corpus quelque part : `corpus : List[ndarray]`.
* `cosine_similarity()` et `add()` ont besoin de connaître $d$ pour valider que les vecteurs ajoutés ont la bonne dimension (éviter d'insérer silencieusement un vecteur de dim 384 dans un corpus de dim 768). D'où `d : int`.
* La normalisation soulève une question de conception. On a vu que si tous les vecteurs sont normalisés, la similarité cosinus se réduit à un simple produit scalaire, ce qui est plus rapide. Deux options :
    * Normaliser à chaque appel de `cosine_similarity()` : correct mais redondant, on recalcule la norme de chaque vecteur du corpus à chaque requête.
    * Normaliser une fois à l'insertion dans `add()`, et stocker les vecteurs déjà normalisés : le calcul de norme est fait $n$ fois en tout, pas $n$ fois par requête.

La seconde option est clairement préférable. Mais on ne veut pas imposer la normalisation sans le dire, certains cas d'usage veulent la distance L2, pas le cosinus. D'où `normalize : bool` comme attribut de configuration, fixé à la construction.

Ici encore on a un problème où l’ont doit maintenir une liste des $k$ meilleurs résultats sans trier tout le corpus (problème top-k). Il est donc naturel d’utiliser ici une structure de donnée *heap* (comme ce qu’on a vu pour TF-IDF).

> Note : cette classe va nous servir de *baseline* ou *ground truth*, ce sera la référence dont on se sert pour comparer et mesurer le rappel@K par rapport à d’autres méthodes. C'est son seul rôle, on ne va pas l'optimiser et sa conception est délibérément minimale. Elle est aussi appelée `IndexFlatIP` dans Faiss (IP = Inner Product = produit scalaire).

```
Classe BruteForceSearch
   Attributs
      corpus    : List[ndarray]     ← liste des embeddings du corpus (chacun de dim d)
      d         : int               ← dimension des vecteurs
      normalize : bool              ← si True, normalise à la construction

   Méthodes
      add(vectors) → None
         Ajoute les vecteurs au corpus
         Si normalize : divise chaque vecteur par sa norme

      cosine_similarity(u, v) → float
         = dot(u, v) / (norm(u) * norm(v))
         = dot(u, v) si vecteurs normalisés

      search(query, k) → List[Tuple[int, float]]
         Pour chaque vecteur v_i du corpus :
            calculer sim_i = cosine_similarity(query, v_i)
         Retourner les k indices avec les sim_i les plus élevés
         Complexité : O(n · d) par requête
```
> **Complexité :** O(n · d) par requête. Pour $n = 1\,000\,000$ documents et $d = 768$ dimensions, cela représente ~768 millions d'opérations par requête. À raison de quelques milliards d'opérations/s sur un CPU moderne, cela donne ~0.1 à 1 seconde par requête, acceptable pour une démo, inutilisable en production à l'échelle.

# PARTIE 3 : La malédiction de la dimensionnalité

## 3.1 Intuition

Nous avons déjà évoqué la [malédiction de la dimensionnalité](https://fr.wikipedia.org/wiki/Fl%C3%A9au_de_la_dimension) dans le cours sur l’ACP en Data Science. Ici cette « malédiction » amène un problème un peu différent mais qui a exactement la même origine.

Notre intuition géométrique est construite en 2 ou 3 dimensions. En haute dimension (768, 1536…), des phénomènes contre-intuitifs apparaissent qui brisent les hypothèses sur lesquelles reposent les structures d'index classiques.

Le phénomène central s'appelle la **concentration des distances** : lorsque la dimension $d$ augmente, les distances entre des points tirés aléatoirement dans $\mathbb{R}^d$ tendent toutes vers la **même valeur**. Le voisin le plus proche et le voisin le plus loin deviennent presque équidistants.

## 3.2 Formalisation

Pour des points tirés uniformément dans la boule unité de $\mathbb{R}^d$, le rapport entre la différence max/min des distances et la distance minimale converge vers 0 :

$$\frac{d_{\max} - d_{\min}}{d_{\min}} \xrightarrow[d \to \infty]{} 0$$

On peut aussi l'exprimer par la **variance relative** des distances : elle s'effondre en $O(1/d)$. En dimension 768, toutes les distances sont concentrées dans un intervalle très étroit autour de leur valeur moyenne $\sqrt{d}$ (qui elle croît, mais l'écart relatif disparaît).

Intuitivement : en dimension $d$, l'immense majorité du volume de la boule unité se trouve dans une coquille sphérique infiniment mince à sa surface. Tous les points sont donc à peu près à la même distance du centre... et les uns des autres.

## 3.3 Illustration numérique

```
Expérience : 500 points aléatoires dans R^d, 1 point requête aléatoire.
On mesure (d_max - d_min) / d_min (ratio de concentration) :

d =    2 :  ratio ≈ 2.41    ← le voisin le plus proche est 3.4× plus proche que le plus loin
d =   10 :  ratio ≈ 0.89
d =   50 :  ratio ≈ 0.28
d =  100 :  ratio ≈ 0.18
d =  200 :  ratio ≈ 0.12
d =  768 :  ratio ≈ 0.06    ← le voisin le plus proche est seulement 6% plus proche que le plus loin

→ En dimension 768, il n'y a plus de structure de proximité exploitable.
```

## 3.4 Conséquence sur les structures d'index exact

Les structures d'index classiques pour la recherche de voisins (**KD-tree** et **Ball Tree**, que nous n’avons pas vu dans ce cours, vous pouvez farie les recherches pour le mémoire par exemple) reposent sur une partition de l'espace en régions hiérarchiques. Leur efficacité vient de la capacité à **éliminer des régions entières** : si la région la plus proche d'un point requête est plus éloignée que le voisin déjà trouvé, toute la région peut être ignorée.

Cette élimination est possible parce qu'en basse dimension, des régions de l'espace sont clairement « dans la mauvaise direction ». En haute dimension, la concentration des distances fait que cette élimination devient impossible : toutes les régions sont à peu près à la même distance, et aucune ne peut être éliminée sans examen.

**KD-tree et Ball Tree dégénèrent en force brute au-delà de ~20 à 50 dimensions.** C'est un résultat bien établi (Bellman, 1961 pour la malédiction de la dimensionnalité ; Weber et al., 1998 pour l'applicabilité aux index).

| Structure | Complexité requête (d petit) | Complexité requête (d = 768) |
|---|---|---|
| Force brute | O(n · d) | O(n · d) |
| KD-tree | O(d · log n) | O(n · d) (dégénérée) |
| Ball Tree | O(d · log n) | O(n · d) (dégénérée) |
| **HNSW (ANN)** | - | **O(d · log n)** |

> **Conclusion :** il faut une approche radicalement différente pour indexer des vecteurs de haute dimension. C'est ce que font les méthodes **ANN** (Approximate Nearest Neighbor).

# PARTIE 4 : Approximate Nearest Neighbor (ANN)

## 4.1 Le compromis précision / vitesse

Les méthodes ANN font un choix délibéré : **accepter de ne pas trouver systématiquement le voisin exact en échange d'un gain de vitesse considérable**.

Le critère de qualité d'un index ANN est le **rappel@K** (*recall@K*) : parmi les $K$ voisins exacts (calculés par force brute), quelle fraction l'index ANN retrouve-t-il ?

$$\text{recall}@K = \frac{|\text{résultats ANN} \cap \text{voisins exacts}|}{K}$$

En pratique, les meilleurs algorithmes ANN atteignent **95–99% de recall@10 avec 100 à 1000 fois de gain en vitesse** par rapport à la force brute.

## 4.2 Pourquoi c'est acceptable en pratique

Dans la plupart des cas d'usage (recherche sémantique, recommandation, RAG), les embeddings du 2e et 3e voisins sont presque aussi pertinents que le premier. Le 2e résultat d'une recherche sémantique est presque toujours pertinent si le 1er l'est. La précision absolue n'est pas requise : ce qui compte, c'est de trouver des résultats **sémantiquement proches** en quelques millisecondes.

## 4.3 Les grandes familles de méthodes ANN

Trois grandes approches coexistent dans la littérature et dans les bibliothèques :

**Hachage sensible à la localité (LSH)** : projeter les vecteurs dans des espaces de plus faible dimension où des vecteurs proches sont mappés sur les mêmes « buckets » avec haute probabilité. Simple à comprendre, mais les performances en rappel restent inférieures aux méthodes par graphes sur les benchmarks modernes.

**Index inversé par quantification (IVF)** : partitionner l'espace vectoriel en $C$ [cellules de Voronoï](https://fr.wikipedia.org/wiki/Diagramme_de_Vorono%C3%AF) (via k-means), puis à la requête n'examiner que les $C'$ cellules les plus proches. Faiss utilise cette approche comme `IndexIVFFlat`. La quantification du produit (PQ) compresse les vecteurs pour réduire la mémoire.
> Rappel : dans un diagramme de Voronoï, chaque cellule enferme un seul germe, et forme l'ensemble des points du plan plus proches de ce germe que d'aucun autre.

**Index par graphe de proximité (HNSW)** : construire un graphe où chaque vecteur est connecté à ses voisins les plus proches, puis naviguer dans ce graphe à la requête. C'est l'approche de référence en terme de rappel/latence, et c'est celle que nous allons détailler (vu que je vous parle de graphe depuis le début du cours).

# PARTIE 5 : HNSW : Hierarchical Navigable Small World

## 5.1 L'inspiration : les graphes « petits mondes »

HNSW (Malkov & Yashunin, 2016) s'inspire d'une propriété observée dans les réseaux réels : la théorie des **petits mondes** (*small world graphs*). Dans un réseau social de millions de personnes, n'importe qui peut atteindre n'importe qui d'autre en seulement 6 « sauts » en moyenne (l'expérience de Milgram, 1967, qui a donné le nom « 6 degrés de séparation »).

Ce phénomène est possible parce que ces réseaux combinent deux types de connexions :
- **Liens locaux** : connexions vers des personnes géographiquement ou professionnellement proches
- **Liens longue portée** : quelques connexions vers des personnes très éloignées, qui permettent de « sauter » rapidement vers une autre région du réseau

La navigation est efficace parce qu'à chaque étape on peut s'approcher du but en choisissant le voisin le plus proche de la cible.

**HNSW transpose exactement ce principe à la recherche de voisins dans un espace vectoriel.**

## 5.2 La structure multi-couches

HNSW construit un **graphe de proximité à plusieurs couches hiérarchiques** :

```
Couche 2 (top)  :   o ──────────────── o       ← peu de nœuds, liens longs
                    │                  │          navigation rapide longue distance
Couche 1        :   o ── o ── o ── o ── o       ← nœuds intermédiaires, liens moyens
                    │    │    │    │    │
Couche 0 (base) :   o─o─o─o─o─o─o─o─o─o─o─o   ← tous les nœuds, liens courts
                                                   couche de précision
```

### Propriétés de chaque couche

- **Couche 0** : contient **tous** les vecteurs du corpus. Chaque nœud est connecté à ses $M$ voisins les plus proches dans l'espace des embeddings. C'est la couche de précision.
- **Couches supérieures (1, 2, …)** : contiennent un **sous-ensemble** de vecteurs, sélectionnés aléatoirement lors de l'insertion. La densité décroît exponentiellement : si la couche $i$ contient $n$ nœuds, la couche $i+1$ en contient en moyenne $n/e$ (avec $e ≈ 2.718$).

Les connexions des couches supérieures sont plus longues (elles relient des vecteurs moins proches) parce qu'il y a moins de nœuds à relier. C'est l'analogue des autoroutes : elles relient des villes éloignées, pas des maisons voisines.

## 5.3 Construction de l'index

Chaque vecteur est inséré un par un. Lors de l'insertion du vecteur $q$ :

1. **Assigner une hauteur maximale** $\ell$ tirée aléatoirement selon une distribution exponentielle décroissante : $P(\ell \geq k) = e^{-k/m_L}$ (avec $m_L$ un paramètre lié à $M$). Cette distribution garantit que la grande majorité des nœuds n'apparaissent qu'en couche 0, et que la proportion diminue exponentiellement avec la hauteur.

2. **Pour chaque couche** de la plus haute jusqu'à la couche 0 :
   - Trouver les $M$ voisins les plus proches de $q$ parmi les nœuds déjà présents dans cette couche (via un BFS guidé)
   - Créer des arêtes bidirectionnelles entre $q$ et ces $M$ voisins

3. **Élaguer si nécessaire** : si l'insertion de $q$ donne à un nœud existant trop de voisins (plus de $M_{\max}$), supprimer les arêtes vers les voisins les moins proches (élagage heuristique).

```
INSÉRER(q, index_HNSW) :

   ℓ ← hauteur_aléatoire()    ← distribution exponentielle décroissante
   ep ← point_entrée_global   ← nœud de départ (le plus haut nœud inséré jusqu'ici)

   // Descendre des couches supérieures jusqu'à ℓ+1 (navigation rapide)
   pour couche = couche_max ... ℓ+1 :
      W ← RECHERCHER_COUCHE(q, ep, ef=1, couche)
      ep ← le plus proche de W par rapport à q

   // De la couche ℓ jusqu'à la couche 0 : connecter q
   pour couche = ℓ ... 0 :
      W ← RECHERCHER_COUCHE(q, ep, ef=ef_construction, couche)
      voisins ← SÉLECTIONNER_VOISINS(q, W, M)    ← sélection heuristique
      Créer les arêtes q ↔ voisins_j pour chaque voisin_j
      Pour chaque voisin_j : élaguer ses connexions si > M_max
      ep ← le plus proche de W par rapport à q (pour la couche suivante)

   si ℓ > couche_max : mettre à jour le point d'entrée global
```

**Complexité de construction :** $O(n \cdot M \cdot \log n)$

## 5.4 Requête : BFS guidé par la distance

Une requête dans HNSW est un **BFS guidé par la distance**, couche par couche. Le lien avec la séance 4 est direct : c'est fondamentalement du BFS, mais la file d'exploration n'est pas FIFO, c'est une **file de priorité** (heap, séance 3) triée par distance, ce qui oriente la navigation vers le but.

L'algorithme descend les couches depuis le sommet :

```
RECHERCHER(q, k, index_HNSW) :

   ep ← point_entrée_global    ← nœud de départ en haut de la hiérarchie

   // Phase 1 : navigation rapide (couches supérieures)
   pour couche = couche_max ... 1 :
      W ← RECHERCHER_COUCHE(q, ep, ef=1, couche)
      ep ← le plus proche de W     ← on met à jour le point d'entrée

   // Phase 2 : recherche précise (couche 0)
   W ← RECHERCHER_COUCHE(q, ep, ef=ef_search, couche=0)

   retourner les k éléments de W les plus proches de q
```

La fonction `RECHERCHER_COUCHE` est le cœur de l'algorithme. Elle maintient deux heaps :

```
RECHERCHER_COUCHE(q, ep, ef, couche) :

   visités ← { ep }
   candidats ← min-heap sur distance(q, ·)   ← candidats à explorer
   résultats ← max-heap sur distance(q, ·)   ← ef meilleurs résultats actuels

   Insérer ep dans candidats et résultats

   tant que candidats non vide :
      c ← extraire le plus proche de candidats
      f ← le plus loin dans résultats

      si distance(q, c) > distance(q, f) :    ← critère d'arrêt
         break                                 ← tous les candidats restants sont plus loin

      pour chaque voisin e de c dans cette couche :
         si e non visité :
            marquer e comme visité
            f ← le plus loin dans résultats
            si distance(q, e) < distance(q, f) ou |résultats| < ef :
               ajouter e à candidats
               ajouter e à résultats
               si |résultats| > ef : supprimer le plus loin de résultats

   retourner résultats
```

**Lien avec les structures de données vues en cours :**
- Le heap `candidats` (min-heap) : toujours extraire le candidat le plus proche à explorer en premier. C'est la file de priorité de la séance 3.
- Le heap `résultats` (max-heap) : maintenir les `ef` meilleurs résultats en expulsant le plus loin quand on en a trop. C'est aussi un heap de la séance 3.
- La navigation couche par couche guidée par la distance la plus courte : c'est du BFS de la séance 4, rendu glouton par la file de priorité.

**Complexité de requête :** $O(d \cdot \log n)$ en moyenne, cette propriété fondamentale rend HNSW efficace à grande échelle.

## 5.5 Les paramètres clés

| Paramètre | Rôle | Valeur typique | Effet d'une augmentation |
|---|---|---|---|
| `M` | Nombre de connexions par nœud et par couche | 16 – 64 | Meilleur rappel, plus de mémoire, construction plus lente |
| `M_max` | Connexions max en couche 0 (souvent = 2M) | 32 – 128 | Idem, pour la couche de précision |
| `ef_construction` | Taille de la file pendant la construction | 100 – 500 | Meilleure qualité du graphe, construction plus lente |
| `ef` (requête) | Taille de la file pendant la recherche | 50 – 200 | Meilleur rappel, requête plus lente |

> **Règle pratique :** `ef` à la requête doit être au minimum égal à `k` (le nombre de voisins demandés). En dessous, on peut manquer des résultats. On choisit `ef` en fonction du compromis rappel/latence souhaité. `M` est le paramètre qui a le plus d'impact sur la mémoire : chaque nœud stocke $M$ connexions, donc l'index HNSW occupe environ $O(n \cdot M \cdot d)$ en mémoire totale.

## 5.6 Architecture de l'index HNSW

```
Classe HNSWIndex
   Attributs
      M               : int                         ← connexions par nœud par couche
      ef_construction : int                         ← taille de file à la construction
      d               : int                         ← dimension des vecteurs
      layers          : List[Dict[int, List[int]]]  ← layers[couche][nœud] = liste de voisins
      vectors         : List[ndarray]               ← les vecteurs du corpus
      entry_point     : int                         ← nœud d'entrée (plus haut niveau)
      max_layer       : int                         ← couche maximale atteinte

   Méthodes
      add(vector) → None
         Insère un nouveau vecteur dans l'index
         Assigne une hauteur aléatoire, connecte le nœud dans chaque couche
         Met à jour entry_point si nécessaire

      search_layer(query, ep, ef, layer) → List[(dist, idx)]
         BFS guidé dans une couche donnée
         Retourne les ef voisins les plus proches
         (algorithme détaillé en 5.4)

      search(query, k, ef=None) → List[(dist, idx)]
         Navigation multi-couches puis recherche précise en couche 0
         Retourne les k voisins les plus proches (approximatifs)

      _distance(u, v) → float
         Distance L2 ou cosinus selon la configuration
```

# PARTIE 6 : Faiss et pgvector : panorama des outils

## 6.1 Faiss (Facebook AI Similarity Search)

Faiss est une bibliothèque open source de Meta AI Research pour la recherche de voisins proches dans des espaces vectoriels de grande dimension. Elle est implémentée en C++ avec des bindings Python, supporte CPU et GPU, et est capable de traiter des corpus de plusieurs milliards de vecteurs.

### Les index principaux

| Index Faiss | Type | Description | Usage |
|---|---|---|---|
| `IndexFlatL2` | Exact | Force brute, distance L2 | Baseline, petits corpus (< 100k) |
| `IndexFlatIP` | Exact | Force brute, produit scalaire | Baseline, vecteurs normalisés |
| `IndexIVFFlat` | ANN | Partitionnement Voronoï | Corpus moyens (100k – 10M) |
| `IndexHNSWFlat` | ANN | Graphe HNSW | Grands corpus, faible latence |
| `IndexIVFPQ` | ANN + compression | Quantification du produit | Très grands corpus, mémoire contrainte |
| `IndexPQ` | Compression seule | Product Quantization | Archivage, recherche grossière |

### Product quantization (PQ) : principe

La quantification du produit est une technique de compression des vecteurs. Au lieu de stocker chaque dimension en float32 (4 octets), on décompose le vecteur en $m$ sous-vecteurs de dimension $d/m$, et on remplace chaque sous-vecteur par l'identifiant du centroïde k-means le plus proche parmi $2^{nbits}$ centroïdes pré-appris.

Résultat : un vecteur de $d = 768$ dimensions en float32 (3072 octets) peut être compressé à quelques dizaines d'octets, avec une perte de précision contrôlée. Cela permet d'indexer des centaines de millions de vecteurs dans la RAM d'une machine standard.

### Workflow Faiss : les étapes standard

Présenter l’usage de `Faiss` et faire le benchmark est une des approches possible pour le mémoire (sujet 9).

## 6.2 pgvector (PostgreSQL)

pgvector est une extension PostgreSQL qui ajoute un type natif `VECTOR(d)` et des opérateurs de similarité, ainsi que la possibilité de créer des index ANN (HNSW ou IVF) directement en SQL. Elle permet de stocker des embeddings dans une base relationnelle et de combiner la recherche vectorielle avec des filtres SQL classiques.

### Le type VECTOR et les opérateurs

| Opérateur | Métrique | Utilisation SQL |
|---|---|---|
| `<->` | Distance L2 | `ORDER BY embedding <-> query_vec` |
| `<=>` | Distance cosinus | `ORDER BY embedding <=> query_vec` |
| `<#>` | Produit scalaire négatif | `ORDER BY embedding <#> query_vec` |

### Workflow pgvector

Présenter l’usage de `pgvector` et faire le benchmark est une des approches possible pour le mémoire (sujet 9).

## 6.4 Cas d'usage emblématique : la boucle RAG

Le cas d'usage industriel le plus répandu des bases vectorielles est le **RAG (Retrieval-Augmented Generation)** : augmenter un LLM (GPT-4, Claude, Llama…) avec une base de connaissances externe, en retrouvant dynamiquement les passages pertinents à chaque question.

```
PIPELINE RAG :

   Phase d'ingestion (une seule fois) :
      Documents bruts
         → Découpage en chunks (paragraphes, fenêtres glissantes)
         → Modèle d'embedding (sentence-transformers, OpenAI...)
         → Vecteurs de dimension d
         → Stockage dans Faiss ou pgvector

   Phase de requête (à chaque question) :
      Question utilisateur
         → Modèle d'embedding (même modèle qu'à l'ingestion)
         → Vecteur de la question
         → Recherche ANN : top-K chunks les plus proches sémantiquement
         → Injection dans le prompt du LLM (contexte + question)
         → LLM génère la réponse en s'appuyant sur les chunks retrouvés
```

C'est ce mécanisme qui permet à un chatbot de « lire » un PDF de 500 pages et de répondre précisément à des questions : la recherche vectorielle retrouve les passages pertinents parmi les milliers de chunks, le LLM formule la réponse à partir de ces passages.

> **Lien avec la séance 6 :** BM25 est souvent utilisé comme premier filtre avant la recherche vectorielle (pipeline hybride). BM25 retrouve rapidement les top-50 candidats par correspondance lexicale, puis les embeddings re-rankent ces 50 candidats pour extraire les 5 plus pertinents sémantiquement. Cette combinaison dépasse les deux approches séparément.

# PARTIE 7 : Pour aller plus loin

## Récapitulatif : choisir la bonne approche de recherche

| Besoin | Approche recommandée |
|---|---|
| Correspondance exacte de termes | Index inversé (séance 6) |
| Ranking par pertinence lexicale | BM25 (séance 6) |
| Recherche sémantique, synonymes | Embeddings + cosinus |
| Grande échelle (> 1M, faible latence) | Faiss HNSW |
| Recherche vectorielle + filtres SQL | pgvector |
| Meilleure précision globale | BM25 (filtrage) + embeddings (re-ranking) |
| Chatbot sur documents privés | Pipeline RAG complet |

## Lien avec les mini-mémoires

### Sujet 8 : *Moteur de recherche textuelle* (connexion directe) et sujet 9 : *Embeddings*.
Comme la séance 7 prolonge directement la séance 6, le sujet 9 est dans la continuité du sujet 8 qui implémente BM25 *from scratch*, où la mise en perspective avec les embeddings (ce que BM25 ne peut pas faire, ce que les embeddings corrigent) constitue une conclusion naturelle de ce travail. Les deux sujets se complètent.

## Questions à maîtriser pour 

## Références


* [Malkov & Yashunin (2018)](https://arxiv.org/abs/1603.09320). Article original HNSW, description complète de la construction et des requêtes
* [Mikolov et al. (2013)](https://arxiv.org/abs/1301.3781). Article Word2Vec, propriété d'analogie vectorielle
* [ANN Benchmarks](https://ann-benchmarks.com). Comparaison objective des algorithmes ANN sur des corpus standardisés
* [Documentation Faiss](https://faiss.ai). Guide officiel, choix d'index, benchmarks, GPU
* [pgvector GitHub](https://github.com/pgvector/pgvector). Installation, types, index, exemples SQL
* [sentence-transformers](https://www.sbert.net). Génération d'embeddings en Python, catalogue de modèles
* [UMAP documentation](https://umap-learn.readthedocs.io). Réduction de dimension pour visualiser des embeddings